In [1]:
#we start with downloading a dataset, for this exercise we chose the tiny shakespeare dataset
!curl -O https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  2483k      0 --:--:-- --:--:-- --:--:-- 2486k


In [1]:
#read the file in to inspect it 
with open("input.txt", "r", encoding='utf-8') as f:
    text=f.read()

In [2]:
#Check the length of the dataset
print("length of dataset in characters is :", len(text))

length of dataset in characters is : 1115394


In [3]:
#All the unique characters that occur in the dataset
#We have to make combinations of these
#This chunk of code is effectively our tokenizer
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(''.join(chars)) #this prints out the list as a string
print(vocab_size)
print(enumerate(chars))
#Space is also included in the cahrs


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
#Create a mapping from characters to integers
#Encoding-Decoding Fucntion
#It simply assigns the index valus of the character in the list-chars

stoi={ch:i for i,ch in enumerate(chars)} #String to Integer Mapping
itos={i:ch for i,ch in enumerate(chars)} #Integer to String Mapping
#Enumerate - Index, Character in a list
encode= lambda s: [stoi[c] for c in s]    #Takes a string and outputs the list of integers; stoi is a dict stoi[c] will always return value associated with that key
decode= lambda l: ''.join([itos[i] for i in l])    #Takes an list of integers and outputs a string

print(encode('hi there'))
print(decode(encode('hi there')))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [5]:
#Tokenization process(This is character wise tokenization)
#Now that we have an encoder and a decoder, we essentially have a tokenizer
#So we can essentially tokenize the entire Shakespeare library
#We use pytorch to do the same
import torch
data=torch.tensor(encode(text), dtype=torch.long)
print(data.shape,data.dtype)
print(data[:1000]) #The first 1000 characters of the Shakespeare dataset will look like this to GPT



torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [6]:
# Let's now split the dataset into train and validation
#we split it 90:10 although 80:20 is the usual one

n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]

In [7]:
#Now we want to feed the text into our transformer to train it
#Feeding the entire dataset is computationally expensive, so we train the transformer on tiny chunks of the dataset
#These chunks have some kind of maximum length- known by different names like block-size, context length

block_size=8
train_data[:block_size+1] #Why +1?- we are simultaneously training the transformer on the chunk itself so +1 would mean that the transformer is trained on the entire block size to predict the next character




tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
#example of above:
#x is the input to the transformer, it will be the first block size characters
#y is just the next block size characters
x=train_data[:block_size]
y=train_data[1:block_size+1] #basically y is offset by 1 w.r.t x and each character in y corresponds to the next character that 'should' come in x, so y is the target
for t in range(block_size): #we are iterating over block_size, t is the time dimension = block_size
    context=x[:t+1] #list of integers including t
    target=y[t] #because for the same index y is offset by 1 w.r.t x
    print(f"When input is  {context} the target is: {target}")
    

When input is  tensor([18]) the target is: 47
When input is  tensor([18, 47]) the target is: 56
When input is  tensor([18, 47, 56]) the target is: 57
When input is  tensor([18, 47, 56, 57]) the target is: 58
When input is  tensor([18, 47, 56, 57, 58]) the target is: 1
When input is  tensor([18, 47, 56, 57, 58,  1]) the target is: 15
When input is  tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
When input is  tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [9]:
#For computational efficiency we use 2-D tensors, block-size and batch size
torch.manual_seed(1337) 
#Computers can't generate truly random numbers, they use a 'seed' number to generate the pseudo random number, fixing the seed will produce the same set of random numbers
#Why-Model weights are initialized randomly, and data is often shuffled randomly. Setting a manual seed means your experiments, results, and training runs become reproducible.
#Why do we need a random number- because we are going to pull chunks from random locations in the dataset
batch_size=4 #How many INDEPENDENT sequences will we process in parallel
#how many tensors are we sending at once, this is done for efficiency reasons as GPUs are extremely capable of handling parallel tasks
block_size=8 #Maximum length of a chunk that will be processed
#4x8 tensor

def get_batch(split):  #it's job is to grab a random chunk of data to feed into the transformer model during training or validation
    data= train_data if split=='train' else val_data #Depending on whether you pass 'train' or 'val' into the function, it chooses whether to pull data from your training dataset (train_data) or your validation dataset (val_data).
    ix=torch.randint(len(data)-block_size, (batch_size,)) #(idx for index)
    #ix is a list of B(batch dimension) random starting positions in the text
    #it may come out as ix =[a,b,c,d] where a,b,c,d are 4 random integers between 0 and len(data)-block_size
    #basically a,b,c,d are the 4 random starting positions
#syntax is torch.randint(high,(size)) high is the max random number- so our choice makes sure that we can atleast grab a full chunk of block_size without running off the end of the file
#(size)= (batch_size,) depicts- 4 random starting points- it is a 1-D tensor as it does not specify other dimensions
    x=torch.stack([data[i:i+block_size] for i in ix]) #slices chunks of length block_cize from data variable and stacks them into a 2-D tensor of size (batch_size,block_size) (4,8)
    y=torch.stack([data[i+1:i+1+block_size] for i in ix]) #offset x by 1
    return x,y  #a tuple is returned
#The targets are in the associated array y which will come all the way at the end to calculate the loss function

xb, yb = get_batch('train')
print("input:")
print("shape of input=", xb.shape)
print(xb)
print("targets:")
print("shape of target=", yb.shape)
print(yb)  #This is present at the end of the forward pass process to calculate the loss function

print("-----")

for b in range(batch_size):  #batch dimension
    for t in range(block_size): #time dimension
        context=xb[b, :t+1] #bth row which is sliced from 0 to t
        target=yb[b,t] #bth row ka tth element- remember y is offset by 1 w.r.t x
        print(f"when input is {context.tolist()} the target is {target}")

input:
shape of input= torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
shape of target= torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] the target is 43
when input is [24, 43] the target is 58
when input is [24, 43, 58] the target is 5
when input is [24, 43, 58, 5] the target is 57
when input is [24, 43, 58, 5, 57] the target is 1
when input is [24, 43, 58, 5, 57, 1] the target is 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is 39
when input is [44] the target is 53
when input is [44, 53] the target is 56
when input is [44, 53, 56] the target is 1
when input is [44, 53, 56, 1] the target is 58
when input is

In [10]:
#Now that we have our input ready we'd like to feed it into a neural network
#The simplest one being Bigram Language Model
import torch
import torch.nn as nn #PyTorch Neural Networks
from torch.nn import functional as F #Functions such as relu, softmax, etc
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self): #We don't need to pass vocab_size in the constructor as it is already defined as a global variable
        super().__init__() #Initializes the parent nn.Module so PyTorch can track weights and layers properly
        self.token_embedding_table= nn.Embedding(vocab_size, vocab_size)
        #PyTorch fills this table with random values from a normal distribution
        #Creates a lookup table where every single character/token in your vocabulary maps directly to a vector of logits for the entire vocabulary size (vocab_size by vocab_size)
        #When we pass idx in the next function every single integer in the input tensor (input tensor is 2-D [4x8]) will pluck out a row with index = integer
    
    def forward(self,idx,targets=None):
        #idx and targets are both (B,T) tensors of integers
        logits=self.token_embedding_table(idx) #(B,T,C)
    #We can interpret logits as the scores for the next character in the sequence
    #We are predicting what comes next based on the individual identity of a single token
    #Logits refers to the raw, unnormalized prediction scores generated by the model before any activation function (like softmax) is applied.
    #(B,T,C) C is the vocabulary size
    #How is it creadted: idx is B,T shaped and when we pass it through embedding table (which is an nn.Embedding(vocab_size, vocab_size) layer), each token integer is used as a lookup index to pull a vector of size C (vocab_size)
        #Calculating Loss
        if targets is None:
            loss=None
        else:
            B,T,C= logits.shape
            logits=logits.view(B*T,C) # .view() is used to reshape the tensor without changing it's underlying data
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits,targets)
        return logits,loss
    #Calculating Loss: If target labels are provided (during training), the 3-D tensor shapes (B, T, C) and (B, T) are reshaped into 2-D and 1-D tensors using .view() so they fit PyTorch's F.cross_entropy requirement. It then calculates the loss (how wrong the predictions are)
    
    #Now we want to be able to generate data from our model, so let's do generation
    def generate(self, idx, max_new_tokens): #Max new tokens = how many new characters you want the model to generate
        for _ in range(max_new_tokens):
            logits,loss=self(idx) #Passes the current context idx into the model (which automatically calls the forward function). It returns logits of shape B,T,C
            #equivalent to self.forward(idx) For context forward function is defined one step earlier
            logits=logits[:,-1,:] #becomes B,C
            #Because a bigram model only looks at the most recent token to predict the next one, so we chose the last element in the time dimension.

            #Now we apply softmax to get probabilities
            probs=F.softmax(logits, dim=-1)
            #Raw logits are unnormalized scores. Passing them through F.softmax turns them into proper probability distributions that sum up to 1 across the vocabulary dimension (C)

            #sample from the distribution
            idx_next=torch.multinomial(probs, num_samples=1) #(B,1)
    #torch.multinomial takes the probability distribution and randomly picks (samples) a token index based on those probabilities. (Tokens with higher probabilities are chosen more often, but there's still a chance for variety). This outputs a tensor of shape (B, 1)
    #That is for each batch we have 1 probability so (B,1(since num_samples=1))
            #Append sampled index to running sequence
            idx=torch.cat((idx,idx_next), dim=1)
            #torch.cat appends the newly generated token (idx_next) onto the end of the existing context idx along the time dimension (dim=1).
            #The sequence length (T) increases by 1 (T+1), so when the for loop runs again, the model now uses this newly grown sequence as its input context.
            #Finally, once the loop finishes generating all the requested tokens, it returns the complete expanded sequence (idx)
        return idx

m=BigramLanguageModel() #m is an instance of BigramLanguageModel 
logits,loss=m(xb,yb)
#m(xb, yb): Passes your input batch (xb) and target batch (yb) directly into the model. In PyTorch, calling an nn.Module instance like a function automatically triggers its forward() method.
print(logits.shape)
print(loss)

print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))
#idx=torch.zeros((1,1), dtype=torch.long : this creates a tensor filled with 0s of shape 1,1 and 64-bit type
#m.generate calls the genrate function with max_new_tokens set to 100 and idx is self- returns a tensor of shape (1,101)
#[0].tolist()-Extracts the first batch row ([0]) and converts the tensor of integer IDs into a standard Python list of numbers.
#decode-Passes that list of integers into your decoding function to convert the numbers back into readable text characters.



torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [11]:
#creating a pytorch optimizer
#optimizer is an algorithm or method used to update model parameters—specifically the weights and biases—in order to minimize the loss function.
optimizer=torch.optim.AdamW(m.parameters(), lr=1e-3) #lr- learning rate a typically good lr is 3e-4 but for much smaller data sets you can have a higher lr


In [12]:
batch_size=32
for steps in range(10000): #for some number of steps (the more the steps the better our optimization- the closer we get to the desired value (however after a certain threshold it may overshoot)

    #sample new batch of data
    xb,yb=get_batch('train')

    #evaluate the loss
    logits,loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True) #instead of filling it with zeroes it fills the .grads to none- it is lighter on the computer memory
    #zeroing out all the gradients form the previous step
    loss.backward()
    optimizer.step()

print(loss.item())

2.382369041442871


In [13]:
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))
#generate function is used to generate output and this time it is a litlle less garbage than the previous genration
#the tokens are not talking to each other, so they must start talking to each other to make better predictions about what comes next

#The neural netwrok can be in two different kinds of 'modes'- evaluation mode and training mode



lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulsee


In [14]:
#SELF ATTENTION    
#Consider the following example
torch.manual_seed(1337)
B,T,C=4,8,2
x=torch.randn(B,T,C) #.randn creates a tensor of the given shape
x.shape

torch.Size([4, 8, 2])

In [15]:
#we want x[b,t]=mean{i<=t} x[b,i]
#version1
xbow=torch.zeros((B,T,C))  #bow-Bag of words- aterm used when averaging out stuff
for b in range(B):
    for t in range(T):
        xprev=x[b,:t+1] #(t,C)
        xbow[b,t]=torch.mean(xprev,0)

#for loop is kind of inefficient so we use matrix multiplication


In [16]:
#version-2
wei=torch.tril(torch.ones(T,T))  #These are our weights- we initialise and then multiply with with the vectors to average them out
wei=wei/wei.sum(1,keepdim=True)
xbow2=wei @ x # wei- (T,T) and @ is a batch multiplier so pytorch creates a batch dimension to match the dimension of x = (B,T,C)-- 
#The result of this multiplication is a (B,T,C)
#and xbow2==xbow (basically the goal was to get xbow in a more efficient way)
#to check if 2 matrices are equal:
torch.allclose(xbow,xbow2)


True

In [17]:
#version-3 using softmax
tril=torch.tril(torch.ones(T,T))
wei=torch.zeros(T,T)
wei=wei.masked_fill(tril==0, float('-inf')) #For all the elements in the matrix where tril==0, make them negative infinity
wei=F.softmax(wei,dim=-1) #Applies probabilities and we get the exact same matrix as before since softmax(-inf)=0
xbow3= wei @ x
torch.allclose(xbow,xbow3)


True

In [18]:
#version-4 
#Self-Attention
torch.manual_seed(1337)
B,T,C=4,8,32
x=torch.randn(B,T,C)

#Let's see a single head perform self attention
head_size=16
key=nn.Linear(C, head_size, bias=False) #the linear module only acts on the last dimension- only the channels so the last dimension becomes 16 as head_size=16
query=nn.Linear(C, head_size, bias=False)
value=nn.Linear(C, head_size, bias=False)
k=key(x)
q=query(x)
#No communication has happened yet, now it comes
wei = q@k.transpose(-2,-1) #we have to transpose the last 2 dimensions for multiplication to take place
#(B,T,16) @ (B,16,T) to give us (B,T,T)

tril=torch.tril(torch.ones(T,T))
#wei=torch.tril(torch.ones(T,T)) #We don't exactly wan't this because different tokens will find other tokens to be more or less interesting or not based on affinities
#Hence we want it to be data dependent
wei=wei.masked_fill(tril==0, float('-inf'))
wei=F.softmax(wei,dim=1)
v=value(x)
out= wei @ v
out.shape


torch.Size([4, 8, 16])

In [19]:
k=torch.randn(B, T, head_size)
q=torch.randn(B, T, head_size)
wei= q@k.transpose(-2,-1)*head_size**-0.5

In [20]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

        


torch.Size([32, 100])